In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1  # Pretrained ImageNet weights (for correct normalization)

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet input size
    transforms.ToTensor(),          # Convert image to tensor
    transforms.Normalize(
        mean=weights.transforms().mean,  # ImageNet mean
        std=weights.transforms().std     # ImageNet std
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),  # Same preprocessing as train
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights.transforms().mean,
        std=weights.transforms().std
    )
])

In [ ]:
data_dir = "cnn_dataset_2"  # Root dataset directory

# Create datasets (ImageFolder expects class subfolders)
train_ds = datasets.ImageFolder(f"{data_dir}/train", transform=train_tfms)
val_ds   = datasets.ImageFolder(f"{data_dir}/valid", transform=val_tfms)
test_ds  = datasets.ImageFolder(f"{data_dir}/test",  transform=val_tfms)

# Create data loaders
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)   # Shuffle for training
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False)    # No shuffle for eval
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False)

class_names = train_ds.classes  # Class labels inferred from folder names
num_classes = len(class_names)  # Number of output classes
print(class_names)

FileNotFoundError: [Errno 2] No such file or directory: 'cnn_dataset_2/train'

In [6]:
from torchvision import datasets
from torch.utils.data import DataLoader, random_split

data_dir = "cnn_dataset_2"

# Load full dataset WITHOUT transform first
full_ds = datasets.ImageFolder(data_dir)

num_classes = len(full_ds.classes)
class_names = full_ds.classes
print(class_names)

# Split ratios
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

total_size = len(full_ds)
train_size = int(train_ratio * total_size)
val_size = int(val_ratio * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    full_ds,
    [train_size, val_size, test_size]
)

# Now assign transforms
train_ds.dataset.transform = train_tfms
val_ds.dataset.transform   = val_tfms
test_ds.dataset.transform  = val_tfms

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False)


['10c', '10d', '10h', '10s', '2c', '2d', '2h', '2s', '3c', '3d', '3h', '3s', '4c', '4d', '4h', '4s', '5c', '5d', '5h', '5s', '6c', '6d', '6h', '6s', '7c', '7d', '7h', '7s', '8c', '8d', '8h', '8s', '9c', '9d', '9h', '9s', 'Ac', 'Ad', 'Ah', 'As', 'Jc', 'Jd', 'Jh', 'Js', 'Kc', 'Kd', 'Kh', 'Ks', 'Qc', 'Qd', 'Qh', 'Qs']


In [7]:
# set up the model
model = resnet18(weights=weights)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, num_classes)

# freeze layers
for param in model.parameters():
    param.requires_grad = False
# unfreeze last layer
for param in model.fc.parameters(): 
    param.requires_grad = True

In [ ]:
def get_free_gpu():
    if not torch.cuda.is_available():
        return torch.device("cpu")  # Fallback to CPU if no GPU

    free_mem = []
    for i in range(torch.cuda.device_count()):
        torch.cuda.set_device(i)
        torch.cuda.empty_cache()  # Clear unused cached memory
        stats = torch.cuda.mem_get_info(i)
        free_mem.append((stats[0], i))

    _, best_gpu = max(free_mem)  # Pick GPU with most free memory
    return torch.device(f"cuda:{best_gpu}")

device = get_free_gpu()
model.to(device)  # Move model to selected device
print("Using device:", device)

criterion = nn.CrossEntropyLoss()  # Multi-class classification loss

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),  # Only trainable params
    lr=1e-3
)

Using device: cuda:7


In [ ]:
def train_one_epoch(model, loader, epoch=None):
    model.train()
    running_loss, correct, total = 0, 0, 0

    pbar = tqdm(loader, desc=f"Epoch {epoch} [train]", leave=False)

    for x, y in pbar:
        x, y = x.to(device), y.to(device)  # Move batch to device

        optimizer.zero_grad()  # Clear previous gradients
        out = model(x)         # Forward pass
        loss = criterion(out, y)
        loss.backward()        # Backprop
        optimizer.step()       # Update weights

        running_loss += loss.item() * x.size(0)  # Accumulate total loss
        preds = out.argmax(1)                    # Predicted class
        correct += (preds == y).sum().item()
        total += y.size(0)

        pbar.set_postfix(
            loss=running_loss / total,  # Running avg loss
            acc=correct / total         # Running accuracy
        )

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()  # Evaluation mode
    correct, total = 0, 0

    for x, y in tqdm(loader, desc="Validation", leave=False):
        x, y = x.to(device), y.to(device)
        out = model(x)
        preds = out.argmax(1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return correct / total  # Final accuracy

In [10]:
# --- Phase 1: Train head only (frozen backbone) ---
for epoch in range(7):  # train 7 epochs first
    train_loss, train_acc = train_one_epoch(
        model, train_loader, epoch=epoch+1
    )
    val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}: "
          f"loss={train_loss:.4f}, "
          f"train_acc={train_acc:.3f}, "
          f"val_acc={val_acc:.3f}")

# --- Phase 2: Unfreeze layer4 for fine-tuning ---
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4  # lower learning rate for fine-tuning
)

for epoch in range(7, 14):  # fine-tune for 7 more epochs
    train_loss, train_acc = train_one_epoch(model, train_loader, epoch=epoch+1)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}: "
          f"loss={train_loss:.4f}, "
          f"train_acc={train_acc:.3f}, "
          f"val_acc={val_acc:.3f}")

Epoch 1: loss=3.2989, train_acc=0.135, val_acc=0.179


Epoch 2: loss=2.8854, train_acc=0.211, val_acc=0.226


Epoch 3: loss=2.7216, train_acc=0.246, val_acc=0.270


Epoch 4: loss=2.6130, train_acc=0.271, val_acc=0.283


Epoch 5: loss=2.5310, train_acc=0.290, val_acc=0.294


...

Epoch 10: loss=0.3622, train_acc=0.922, val_acc=0.907


Epoch 11: loss=0.2414, train_acc=0.956, val_acc=0.945


Epoch 12: loss=0.1591, train_acc=0.976, val_acc=0.960


Epoch 13: loss=0.1071, train_acc=0.986, val_acc=0.971


Epoch 14: loss=0.0734, train_acc=0.993, val_acc=0.974


In [ ]:
def evaluate_test(model, loader):
    model.eval()  # Evaluation mode
    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():  # Disable gradient tracking
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()  # Accumulate batch loss
            _, preds = torch.max(outputs, 1)  # Predicted class indices

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total  # Avg loss, accuracy


# Run test
test_loss, test_acc = evaluate_test(model, test_loader)

print(f"\nFinal Test Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.3f}")


Final Test Results:
Test Loss: 0.1240
Test Accuracy: 0.973


In [12]:
torch.save(model.state_dict(), "cnn_resnet18_224x224_3.pt")
